Start time of execution

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymongo
import yaml
from time import time

from utils import sliding_window_pd, filter_instances, check_missing_values
from utils_visual import (
    plot_filtered_vs_raw_signal,
    plot_feature_boxplots_by_class,
    plot_scatter_pca,
    plot_window_counts_by_gesture_user
)
from utils_features import extract_all_candidates

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

**Load Configuation**

In [ ]:
config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [ ]:
time_start = time()


In [ ]:
client = pymongo.MongoClient(config["client"])
db = client[config["db"]]
collection = db[config["col"]]


In [ ]:
found_keys = collection.distinct("_id")
print("Total number of documents in the collection:", len(found_keys))

In [ ]:
documents = collection.find({"_id": {"$in": found_keys}})
rows = []
for doc in documents:
    n_samples = len(doc["data"]["acc_x"])
    for i in range(n_samples):
        rows.append({
            "acc_x": float(doc["data"]["acc_x"][i]),
            "acc_y": float(doc["data"]["acc_y"][i]),
            "acc_z": float(doc["data"]["acc_z"][i]),
            "gyr_x": float(doc["data"]["gyr_x"][i]),
            "gyr_y": float(doc["data"]["gyr_y"][i]),
            "gyr_z": float(doc["data"]["gyr_z"][i]),
            "gesture_id": doc["gesture_id"],

            "user": doc["user"],
        })

dataframe = pd.DataFrame(rows)
# Should be float/int
print(dataframe[["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z", "gesture_id","user"]].dtypes)
print(dataframe[:5])

## Data Exploration
This section will explore the dataset to understand its structure, identify any missing values, and analyze the distribution of the target variable.

### Step 1: Barplot of each recording

In [ ]:
from utils_visual import plot_gesture_duration_by_user
plot_gesture_duration_by_user(
    df = dataframe
)


### Step 2: Detect Missing Values

In [ ]:
from utils import check_missing_values
check_missing_values(dataframe)

### Step 3: Filter Signals

In [ ]:
list_of_signals = [value for key,value in dataframe.groupby(["gesture_id","user"])]
filtered_list_of_signals = filter_instances(
        instances_list = list_of_signals,
        order=config["filter"]["order"],
        filter_type=config["filter"]["type"],
        wn = config["filter"]["wn"]
    )

In [ ]:
plot_filtered_vs_raw_signal(
    raw_signals = list_of_signals,
    filtered_signals = filtered_list_of_signals,
    idx = 0,
    n_start = 0,
    n_end = 100
)

### Step 4: Chronological Pre-Window Split
Instead of splitting the windows randomly, we split the continuous recording sessions by **chronological order** first (75% training, 25% testing) before windowing.

This ensures that the training and testing sets are temporally disjoint, completely preventing overlapping windows (data leakage) across the split boundary.

In [ ]:

train_signals = [session [:22500] for session in filtered_list_of_signals] # 22500 is the 75% of the total samples
test_signals = [session [22500:] for session in filtered_list_of_signals]
train_windows = []
test_windows = []
for signal in train_signals:
        temp_windows = []
        temp_windows = sliding_window_pd(
            df=signal,
            ws=config["sliding_window"]["ws"],
            overlap=config["sliding_window"]["overlap"],
            w_type=config["sliding_window"]["w_type"],
        )
        
        for window in temp_windows:
            train_windows.append(window)
for signal in test_signals:
    temp_windows = sliding_window_pd(
        df=signal,
        ws=config["sliding_window"]["ws"],
        overlap=config["sliding_window"]["overlap"],
        w_type=config["sliding_window"]["w_type"],
    )
    for window in temp_windows:
        test_windows.append(window)

windows = train_windows + test_windows
print(f"Total training windows: {len(train_windows)}")
print(f"Total testing windows: {len(test_windows)}")


### Step 5: Windows Barplot
Windowing Barploting in respect to class distribution and user distribution. That helps us to identify if a user and a specific activity is overrepresented in the dataset.

In [ ]:
plot_window_counts_by_gesture_user(windows)


### Data Transformation and Feature Engineering
Total Pipeline of Feature Engineering and Data Transformations steps:
1. For each window we will compute exhaustive set of features (statistical, time-domain, frequency-domain, etc.) (function `extract_all_candidates`).
2. For each train window we will aply ANOVA test to select the most relevant features (parameter `k` of `SelectKBest`).
3. Drawing correlation heatmap of the selected features to identify any multicollinearity issues and understand the relationships between features.
4. We will draw pairwise scatter plots of the selected features to visualize their distribution and potential separability between classes. Also, verifies correlation heatmap results.
5. Based on the correlation heatmap and pairwise scatter plots, we will identify and remove any highly correlated features to reduce redundancy and improve model performance.
6. We will apply standard scaling to the selected features to ensure that they are on the same scale, which is important for many machine learning algorithms.
7. We will apply PCA to the scaled features to reduce dimensionality and visualize the data in a 2D space. This can help us to understand the underlying structure of the data and identify any potential clusters or patterns.

In [ ]:
features_df_train = pd.DataFrame([extract_all_candidates(window) for window in train_windows])
features_df_test = pd.DataFrame([extract_all_candidates(window) for window in test_windows])
print("Extracted features shape:", features_df_train.shape)
print("First few rows of the features dataframe:")
print(features_df_train.head())

In [ ]:
X_train = features_df_train.drop(columns=["gesture_id","user"])
y_train = features_df_train["gesture_id"]
X_test = features_df_test.drop(columns=["gesture_id","user"])
y_test = features_df_test["gesture_id"]

In [ ]:
selector = SelectKBest(f_classif, k=10)
selector.fit(X_train, y_train)
scores   = pd.Series(selector.scores_, index=X_train.columns).sort_values(ascending=False)
selected = X_train.columns[selector.get_support()].tolist()
print("score: ")
print(scores.head(10))
print("ANOVA F-scores (top 10):")
print(selected)

In [ ]:
from utils_visual import plot_correlation_matrix
plot_correlation_matrix(
    df= X_train[selected],
    filename="stratified.png"

)

### Pairwise Scatter Plots

In [ ]:
from utils_visual import plot_pairwise_scatter

# Combine selected features and gesture labels for plotting
X_tr_with_labels = X_train[selected].copy()
X_tr_with_labels["gesture_id"] = y_train.values

plot_pairwise_scatter(
    df=X_tr_with_labels,
    features=selected,
    label_col="gesture_id",
    display_type="grid",
    filename="pairwise_scatter_temporal_split.png"
)


### Outlier Analysis
We generate boxplots of the selected features grouped by gesture category to inspect their distribution and identify any potential outliers. 

> [!NOTE]
> In IMU sensor data, outliers (extreme values) often correspond to rapid physical transitions or high-acceleration peaks during gesture execution. As these are natural features of the gesture rather than sensor errors, we do not remove or clip them, as doing so could degrade classifier performance.

In [ ]:
plot_feature_boxplots_by_class(
    df=pd.concat([X_train[selected], y_train], axis=1),
    features=selected,
    label_col="gesture_id",
    figsize_per_plot=(6, 5),
    filename="boxplot_by_class_termporal_split.png"
)

# Feature Selection Strategy

To optimize model stability and eliminate redundancy, we applied a two-stage feature selection process combining **collinearity analysis**, **ANOVA $F$-testing**, and **sensor domain knowledge**.

### 1. The Multicollinearity Cluster
The initial correlation matrix revealed a severe multicollinearity cluster ($|r| \geq 0.81$) among the time-domain accelerometer features: `acc_y_mean`, `acc_y_rms`, `acc_mag_mean`, and `acc_sma`. Keeping all of them introduces heavy redundancy that destabilizes model training.

### 2. Selection Criteria: ANOVA vs. Domain Knowledge
To break the tie within this cluster, we evaluated two dimensions:
* **Statistical Power:** Univariate ANOVA $F$-tests revealed that `acc_y_rms` is the strongest individual predictor ($F$-score: **619.88**), followed by `acc_mag_mean` (466.85) and `acc_y_mean` (461.47).
* **Domain Knowledge:** * **RMS vs. Mean:** While `acc_y_mean` captures only static bias/gravity alignment, `acc_y_rms` (Root Mean Square) measures the **total signal energy** (both dynamic motion and static components). For physical gesture tracking, signal power is highly descriptive.
  * **Y-Axis Dominance:** The extreme correlation between the Y-axis metrics and the overall magnitude (`acc_mag_mean`) proves the movement is primarily executed along the Y-axis, rendering global magnitude redundant.

---

### Final Dropping Decisions

| Feature | Action | Justification |
| :--- | :---: | :--- |
| **`acc_y_rms`** | **KEEP** | **Top Predictor.** Highest $F$-score (619.88); captures overall physical energy along the dominant axis. |
| **`acc_y_mean`** | ❌ **DROP** | **Redundant.** Highly collinear ($r = 0.96$) with `acc_y_rms` and lower $F$-score. |
| **`acc_mag_mean`** | ❌ **DROP** | **Redundant.** Highly collinear ($r = 0.92$) with `acc_y_rms`; redundant due to Y-axis spatial dominance. |
| **`acc_sma`** | ❌ **DROP** | **Redundant.** Weakest predictor in the cluster ($F$-score: 288.70) and structurally repetitive. |
| **`acc_y_skew`** | **KEEP** | **Unique Signal.** Low correlation with the cluster; tracks structural asymmetry in the movement. |
| **Cross-Sensor Corrs** | **KEEP** | **High Value.** Features like `corr_acc_y_gyr_x` show high predictive power ($F$-score: 453.52) without collinearity ($|r| < 0.5$). |

In [ ]:
cols_to_drop = ["acc_y_mean", "acc_mag_mean","acc_sma"]
X_tr_reduced = X_train[selected].drop(columns=cols_to_drop)
X_te_reduced = X_test[selected].drop(columns=cols_to_drop)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_reduced)
X_te_scaled = scaler.transform(X_te_reduced)

In [ ]:
X_tr_pca = PCA(n_components=3, random_state=42).fit_transform(X_tr_scaled)
pca_df = pd.DataFrame(X_tr_pca, columns=["PC1", "PC2", "PC3"])
binary_labels = y_train.astype("category").cat.codes
pca_df["binary_label"] = binary_labels.values
from utils_visual import plot_scatter_pca
plot_scatter_pca(
    df=pca_df,
    c_name="binary_label",
    filename="pca_chronological_split.png"
)

In [ ]:
svc = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svc.fit(X_tr_scaled, y_train)
y_pred_svc = svc.predict(X_te_scaled)
random_forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
random_forest.fit(X_tr_reduced, y_train)
y_pred_rf = random_forest.predict(X_te_reduced)
from sklearn.metrics import classification_report
print("SVC Classification Report:")
print(classification_report(y_test, y_pred_svc))
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))


In [ ]:
from utils_visual import plot_confusion_matrix
confusion_svc = confusion_matrix(y_test, y_pred_svc,labels=svc.classes_)
plot_confusion_matrix(
    cm = confusion_svc,
    filename = "confusion_svc_temporal_split.png",
    output_dir = "figures/evaluation/ml",
    labels = svc.classes_
)

### Experiment evaluation
The subject-independent split performed very poorly, proving that handcrafted features struggle to generalize to unseen users due to differences in coordinate orientation and physical execution styles. Conversely, the stratified random split yielded an inflated accuracy of around 98%. This massive difference is not a sign of real-world success, but rather the result of temporal data leakage. Because adjacent sliding windows overlap by 50%, a random split forces the training and testing sets to share highly correlated raw sensor readings.

To measure the true generalizability on known users, the chronological split partitioned the timeline before windowing. This removed all overlapping leakage, dropping the accuracy to a realistic 63%–67%. This final baseline reveals that while the model retains user-specific patterns, it must still grapple with chronological session drift, body fatigue, and sensor slippage over time.

### Cross Validation on 2nd split
To further validate the performance of our model on the 2nd split, we will perform cross-validation. This will involve splitting the training data into multiple folds and training the model on different combinations of these folds while evaluating on the remaining fold. This process will be repeated multiple times to ensure that our results are consistent and not due to random chance. We will report the average performance metrics across all folds to provide a more robust estimate of our model's performance on unseen data.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.metrics import make_scorer, accuracy_score
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
param_grid = {
    "rf__n_estimators": [100, 200],
    "rf__max_depth": [None, 10, 20],
    "rf__min_samples_leaf": [1, 3, 5]
}

pipeline_steps = [
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1)
    )
]
grid = GridSearchCV(Pipeline(pipeline_steps), param_grid,cv=5,scoring='accuracy', n_jobs=-1)
grid.fit(X_tr_reduced, y_train)
y_pred_grid = grid.predict(X_te_reduced)
print("Best parameters found: ", grid.best_params_)
print("Classification Report for Random Forest (Grid Search):")
print(classification_report(y_test, y_pred_grid))
confusion_grid = confusion_matrix(y_test, y_pred_grid,labels=grid.best_estimator_.named_steps['rf'].classes_)
plot_confusion_matrix(
    cm=confusion_grid,
    labels = grid.best_estimator_.named_steps['rf'].classes_,
    output_dir = "figures/evaluation/ml",
    filename="rf_temporal_split_tuned.png"
)

In [ ]:
time_end = time()
elapsed_time = time_end - time_start
print(f"Total execution time: {elapsed_time:.2f} seconds")